Prueba

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [2]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\OneDrive\\Escritorio\\computadorNuevo\\SNconsumptionFinal.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [3]:
datos.head()

,temp,zone1,zone2,zone3,hour
date,,,,,
2017-01-01 00:00:00,6.559,34055.69620,16128.87538,20240.96386,0
2017-01-01 00:10:00,6.414,29814.68354,19375.07599,20131.08434,0
2017-01-01 00:20:00,6.313,29128.10127,19006.68693,19668.43373,0
2017-01-01 00:30:00,6.121,28228.86076,18361.09422,18899.27711,0
2017-01-01 00:40:00,5.921,27335.69620,17872.34043,18442.40964,0


In [4]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Se divide el dataset

In [5]:
# Dividir el conjunto de datos en entrenamiento y prueba
train, test = train_test_split(datos, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
test, val = train_test_split(test, test_size=0.33, shuffle=False)

print("Las dimensiones de train son: ", train.shape)
print("Las dimensiones de test son: ", test.shape)
print("Las dimensiones de val son: ", val.shape)

Las dimensiones de train son:  (36691, 5)
Las dimensiones de test son:  (10535, 5)
Las dimensiones de val son:  (5190, 5)


Se normalizan los datos

In [6]:
from sklearn.preprocessing import StandardScaler


# Normalizar solo con los datos de entrenamiento
scaler = StandardScaler()
train = scaler.fit_transform(train)

# Aplicar la transformación a test y val usando los parámetros de train
test = scaler.transform(test)
val = scaler.transform(val)

train = pd.DataFrame(train, columns=datos.columns)
test = pd.DataFrame(test, columns=datos.columns)
val = pd.DataFrame(val, columns=datos.columns)


Se unen los datos nuevamente, ahora normalizados, en un único conjunto.

In [7]:
datosNormalizados = pd.concat([train, test, val])

datosNormalizados.index = datos.index


In [8]:
datosNormalizados.shape

(52416, 5)

In [9]:
datosNormalizados.head(10)


,temp,zone1,zone2,zone3,hour
date,,,,,
2017-01-01 00:00:00,-2.051356,0.151445,-0.851669,0.033500,-1.660858
2017-01-01 00:10:00,-2.074813,-0.433079,-0.211097,0.016502,-1.660858
2017-01-01 00:20:00,-2.091152,-0.527709,-0.283791,-0.055068,-1.660858
2017-01-01 00:30:00,-2.122213,-0.651648,-0.411186,-0.174054,-1.660858
2017-01-01 00:40:00,-2.154568,-0.774750,-0.507632,-0.244729,-1.660858
2017-01-01 00:50:00,-2.165569,-0.872729,-0.597600,-0.293039,-1.660858
2017-01-01 01:00:00,-2.199865,-0.958984,-0.681090,-0.321667,-1.516340
2017-01-01 01:10:00,-2.223323,-1.035189,-0.746587,-0.396816,-1.516340
2017-01-01 01:20:00,-2.193880,-1.127306,-0.832236,-0.463913,-1.516340


Espacio de búsqueda

In [10]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas GRU
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades GRU
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes GRU, es decir [observaciones, retardos, caracteristicas]

In [11]:
futuros = 1
pasados  = 12

In [12]:
datosX = []
datosY = []
for i in range(pasados, len(datosNormalizados) - futuros + 1):
  datosX.append(datosNormalizados.iloc[i-pasados:i, 0:datosNormalizados.shape[1]])
  datosY.append(datosNormalizados.iloc[i+futuros-1:i+futuros, 1])


In [13]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (52404, 12, 5)
Dimensiones de Y: (52404, 1)


In [14]:
inputs = datosX.shape[1] * datosX.shape[2]
datosX = datosX.reshape(datosX.shape[0], inputs)

In [15]:
print("Dimensiones de X después de rehape:", datosX.shape)

Dimensiones de X después de rehape: (52404, 60)


Se dividen nuevamente los conjuntos de datos

In [16]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (36682, 60)
Las dimensiones de testX son:  (10533, 60)
Las dimensiones de valX son:  (5189, 60)


In [17]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (36682, 1)
Las dimensiones de testY son:  (10533, 1)
Las dimensiones de valY son:  (5189, 1)


Se crean métricas para medir desempeño

In [18]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [19]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [20]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(trainX.shape[1],)))
    if (params['layers'] == 1):
      model.add(Dense(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(Dense(units=params['units'], activation=params['activation']))
          model.add(Dropout(params['dropout']))
      model.add(Dense(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(trainX, trainY, epochs=128,
                        validation_data=(testX, testY),
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [21]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=12, trials=trials, rstate=np.random.default_rng(42))

print(best)

  0%|          | 0/12 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

287/287 - 8s - 28ms/step - ia: 0.2674 - loss: 1.2491 - mae: 0.9158 - rmse: 1.1148 - smape: 1.4796 - val_ia: 0.2424 - val_loss: 0.8193 - val_mae: 0.7497 - val_rmse: 0.9000 - val_smape: 1.6552

Epoch 2/128                                           

287/287 - 1s - 3ms/step - ia: 0.2366 - loss: 1.0978 - mae: 0.8649 - rmse: 1.0462 - smape: 1.5214 - val_ia: 0.2425 - val_loss: 0.8045 - val_mae: 0.7446 - val_rmse: 0.8909 - val_smape: 1.8059

Epoch 3/128                                           

287/287 - 1s - 4ms/step - ia: 0.2392 - loss: 1.0268 - mae: 0.8350 - rmse: 1.0117 - smape: 1.5088 - val_ia: 0.2749 - val_loss: 0.7508 - val_mae: 0.7170 - val_rmse: 0.8602 - val_smape: 1.6341

Epoch 4/128                                           

287/287 - 2s - 6ms/step - ia: 0.2840 - loss: 0.9278 - mae: 0.7889 - rmse: 0.9619 - smape: 1.4337 - val_ia: 0.3496 - val_loss: 0.6458 - val_mae: 0.6593 - val_rmse: 0.7968 - val_smape: 1.3620

Epoch 5/128

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                       

2293/2293 - 14s - 6ms/step - ia: 0.9546 - loss: 0.0155 - mae: 0.0710 - rmse: 0.0921 - smape: 0.2015 - val_ia: 0.7545 - val_loss: 0.0060 - val_mae: 0.0624 - val_rmse: 0.0737 - val_smape: 0.1916

Epoch 2/128                                                                       

2293/2293 - 9s - 4ms/step - ia: 0.9641 - loss: 0.0062 - mae: 0.0560 - rmse: 0.0727 - smape: 0.1739 - val_ia: 0.7403 - val_loss: 0.0103 - val_mae: 0.0783 - val_rmse: 0.0897 - val_smape: 0.2390

Epoch 3/128                                                                       

2293/2293 - 8s - 4ms/step - ia: 0.9674 - loss: 0.0052 - mae: 0.0509 - rmse: 0.0667 - smape: 0.1612 - val_ia: 0.8506 - val_loss: 0.0031 - val_mae: 0.0381 - val_rmse: 0.0496 - val_smape: 0.1188

Epoch 4/128                                                                       

2293/2293 - 9s - 4ms/step - ia: 0.9686 - loss: 0.0049 - mae: 0.0488 - rmse: 0.0643 -

...

Epoch 1/128                                                                           

2293/2293 - 19s - 8ms/step - ia: 0.6644 - loss: 0.4454 - mae: 0.5111 - rmse: 0.6311 - smape: 0.8689 - val_ia: 0.4139 - val_loss: 0.1454 - val_mae: 0.3026 - val_rmse: 0.3236 - val_smape: 0.6452

Epoch 2/128                                                                           

2293/2293 - 9s - 4ms/step - ia: 0.8105 - loss: 0.1417 - mae: 0.2939 - rmse: 0.3675 - smape: 0.5918 - val_ia: 0.5186 - val_loss: 0.0629 - val_mae: 0.1995 - val_rmse: 0.2193 - val_smape: 0.4951

Epoch 3/128                                                                           

2293/2293 - 10s - 4ms/step - ia: 0.8599 - loss: 0.0790 - mae: 0.2176 - rmse: 0.2744 - smape: 0.4780 - val_ia: 0.5852 - val_loss: 0.0387 - val_mae: 0.1527 - val_rmse: 0.1716 - val_smape: 0.3991

Epoch 4/128                                                                           

2293/2293 - 6s - 3ms/step - ia: 0.8814 - loss: 0.0575 - mae: 0.1846

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                            

4586/4586 - 15s - 3ms/step - ia: 0.2428 - loss: 6.1339 - mae: 1.7251 - rmse: 2.3036 - smape: 1.5843 - val_ia: 0.1086 - val_loss: 2.9219 - val_mae: 1.3723 - val_rmse: 1.3936 - val_smape: 1.4881

Epoch 2/128                                                                            

4586/4586 - 10s - 2ms/step - ia: 0.2575 - loss: 4.7624 - mae: 1.5512 - rmse: 2.0452 - smape: 1.5651 - val_ia: 0.1136 - val_loss: 2.4817 - val_mae: 1.2723 - val_rmse: 1.2937 - val_smape: 1.4906

Epoch 3/128                                                                            

4586/4586 - 10s - 2ms/step - ia: 0.2686 - loss: 3.8076 - mae: 1.4130 - rmse: 1.8386 - smape: 1.5601 - val_ia: 0.1155 - val_loss: 2.1434 - val_mae: 1.1890 - val_rmse: 1.2100 - val_smape: 1.4956

Epoch 4/128                                                                            

4586/4586 - 10s - 2ms/step - ia: 0.2803 - loss: 3.0000 - mae: 

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                            

287/287 - 4s - 15ms/step - ia: 0.7387 - loss: 0.2809 - mae: 0.4100 - rmse: 0.5136 - smape: 0.7567 - val_ia: 0.8299 - val_loss: 0.0906 - val_mae: 0.2389 - val_rmse: 0.2872 - val_smape: 0.5680

Epoch 2/128                                                                            

287/287 - 2s - 5ms/step - ia: 0.8275 - loss: 0.1318 - mae: 0.2867 - rmse: 0.3619 - smape: 0.5885 - val_ia: 0.8894 - val_loss: 0.0372 - val_mae: 0.1503 - val_rmse: 0.1880 - val_smape: 0.4205

Epoch 3/128                                                                            

287/287 - 1s - 4ms/step - ia: 0.8519 - loss: 0.0965 - mae: 0.2453 - rmse: 0.3095 - smape: 0.5331 - val_ia: 0.9025 - val_loss: 0.0289 - val_mae: 0.1321 - val_rmse: 0.1668 - val_smape: 0.3750

Epoch 4/128                                                                            

287/287 - 1s - 4ms/step - ia: 0.8700 - loss: 0.0748 - mae: 0.2150 - rm

In [22]:
print(best)

{'activation': 3, 'batch': 1, 'dropout': 0.0, 'layers': 4.0, 'learning_rate': 0.00027101707182655693, 'units': 4}
